# 15 — Concentration-to-Mega-Cap Mechanism Bridge

This notebook addresses the main remaining identification gap in the paper: whether the observed mega-cap direct-weight mechanism is actually stronger when aggregate market concentration is higher.

## Frozen methodology

Before viewing results, two primary bridge questions are fixed:

1. `DirectWeightEffect_t ~ Top10Share_t`
2. `(Top10Weight_Q5,t - Top10Weight_Q1,t) ~ Top10Share_t`

Top-10 share is scaled per +10 percentage points. HAC6 is primary and HAC12 is robustness. Because concentration is highly persistent and trending, a linear time-trend specification is reported as a secondary robustness check. No alternative Top-N, HHI, nonlinear terms, sample splits, or lag search are introduced here.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel, build_market_state_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import quintile_returns, quintile_return_panel
from src.concentration import calculate_monthly_concentration
from src.mega_cap import exclude_top_permco
from src.mega_cap_decomposition import fixed_rank_value_weighted_returns, fixed_rank_spread, decomposition_table, mega_cap_leg_diagnostics
from src.concentration_mechanism_bridge import build_weight_gap, run_bridge_regressions, bridge_summary


## 1. Rebuild the frozen research objects


In [ ]:
MONTHLY = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'

clean, _ = build_clean_monthly_panel(MONTHLY, CleaningConfig())
history = build_monthly_history_panel(MONTHLY)
market_state = build_market_state_panel(MONTHLY)
concentration = calculate_monthly_concentration(market_state)

mom = assign_signal_quintiles(
    attach_momentum_to_formation_universe(clean, history)
)
baseline = quintile_return_panel(
    quintile_returns(mom), return_col='VW_Return'
)

ex10_clean = exclude_top_permco(clean, market_state, n=10)
ex10_mom = assign_signal_quintiles(
    attach_momentum_to_formation_universe(ex10_clean, history)
)
ex10 = quintile_return_panel(
    quintile_returns(ex10_mom), return_col='VW_Return'
)

fixed_qret = fixed_rank_value_weighted_returns(mom, market_state, n=10)
fixed = fixed_rank_spread(fixed_qret)
decomp = decomposition_table(baseline, fixed, ex10)
leg_diag = mega_cap_leg_diagnostics(mom, market_state, n=10)
weight_gap = build_weight_gap(leg_diag)

print('Decomposition months:', len(decomp))
print('Weight-gap months:', len(weight_gap))
print('Concentration months:', len(concentration))


## 2. Descriptive bridge diagnostics


In [ ]:
bridge_descriptive = (
    decomp[['MthCalDt','DirectWeightEffect']]
    .merge(weight_gap, on='MthCalDt', how='inner', validate='one_to_one')
    .merge(concentration[['MthCalDt','Top10Share']], on='MthCalDt', how='inner', validate='one_to_one')
)

bridge_descriptive[['DirectWeightEffect','Top10WeightGap_Q5MinusQ1','Top10Share']].describe(percentiles=[0.01,0.05,0.5,0.95,0.99])


In [ ]:
bridge_descriptive[['DirectWeightEffect','Top10WeightGap_Q5MinusQ1','Top10Share']].corr()


## 3. Frozen bridge regressions


In [ ]:
results = run_bridge_regressions(
    decomposition=decomp,
    weight_gap=weight_gap,
    concentration=concentration,
)
summary = bridge_summary(results)
summary


## 4. Primary coefficients only


In [ ]:
summary[['Specification','Estimate_per_10pp','HAC_SE','T_Stat','P_Value','CI95_Low','CI95_High','N','R2']]


## Interpretation gate

The result should be interpreted according to the pre-specified questions, not according to statistical significance alone.

- If concentration positively predicts the direct-weight effect and/or the Q5-minus-Q1 mega-cap weight gap, the paper can more directly connect rising aggregate concentration to the portfolio-implementation mechanism.
- If the coefficients are weak or disappear with the time trend, the paper should state that mega-cap composition matters for value-weighted momentum but that the analysis does not establish a mechanically stronger distortion in high-concentration states.

Do not add alternative thresholds or transformations after seeing this result.
